# 05 · Transfer learning: a pretrained ResNet18 (deliverable 3)

A dense network flattens the painting and loses its 2-D structure (S05, S07). A **convolutional** network looks at small neighbourhoods with the same filters everywhere, and builds edges → textures → object parts → objects, layer after layer (S07).
With only ~2 300 training images we do not train a CNN from scratch: we start from a **ResNet18 pretrained on ImageNet** (1.2 million photos) and adapt it to our genres, as in TP07 Part D:

1. replace its last layer by a new one for our classes, **freeze** the rest and train the **head** only;
2. paintings are very different from ImageNet photos (S07, transfer-learning table): **unfreeze the last block** (`layer4`) and **fine-tune** it with a smaller learning rate;
3. compare both on validation, measure the chosen one **once** on the test set, with its CPU cost for the kiosk.

## 0. Setup

In [ ]:
import copy, json, math, sys, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

sys.path.insert(0, "../src")                 # the team's shared code
from museum_guide.data import prepare_subset

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

ROOT = Path("..")
IMG_SIZE = 128            # our images are 128x128 (TP07 uses 64 on a CPU; 224 = ImageNet, more accurate but slower)
FIG_DIR = ROOT / "results" / "figures"; FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR = ROOT / "results"
MODEL_DIR = ROOT / "data" / "models"; MODEL_DIR.mkdir(parents=True, exist_ok=True)    # weights stay out of Git
print("device:", device)

## 1. Prepare the images like the pre-training
*"A pretrained model must always receive inputs prepared like during its pre-training"* (TP07): resize, then normalise each channel with the **ImageNet** mean (0.485, 0.456, 0.406) and std (0.229, 0.224, 0.225) (S07).
`prepare_subset` gives the team's split (same images as every other model) but standardised with *our* statistics: we undo that first, back to pixels in [0, 1].

In [ ]:
data = prepare_subset("genre", seed=SEED)
class_names = data["classes"]
C = len(class_names)
idx = {"train": data["train_idx"], "val": data["val_idx"], "test": data["test_idx"]}

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def prepare(X):
    X = X * data["std"] + data["mean"]                                           # undo the team standardisation: [0, 1]
    X = F.interpolate(X, size=IMG_SIZE, mode="bilinear", align_corners=False)    # resize
    return (X - IMAGENET_MEAN) / IMAGENET_STD                                    # ImageNet normalisation


X_img = prepare(data["X"])
y_all = data["y"]
data["X"] = None                                                                 # free the memory

full_set = TensorDataset(X_img, y_all)
g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(Subset(full_set, idx["train"]), batch_size=64, shuffle=True, generator=g)
val_loader = DataLoader(Subset(full_set, idx["val"]), batch_size=128)
test_loader = DataLoader(Subset(full_set, idx["test"]), batch_size=128)

print(C, "classes:", class_names)
print("images:", tuple(X_img.shape), "|", {s: len(i) for s, i in idx.items()})
print("train mean per channel:", X_img[torch.as_tensor(idx["train"])].mean(dim=(0, 2, 3)).round(decimals=2).tolist())